In [1]:
import kagglehub
path = kagglehub.dataset_download("reihanenamdari/breast-cancer")

100%|██████████| 42.8k/42.8k [00:00<00:00, 27.5MB/s]

Extracting files...


In [9]:
!pip install --ignore-installed blinker
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 37.9 MB/s eta 0:00:00


In [10]:
!pip install --ignore-installed blinker
!pip install streamlit

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)


In [11]:
!pip install streamlit pyngrok

In [12]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [2]:
import os

# List the contents of the downloaded directory
print(os.listdir(path))

['Breast_Cancer.csv']


In [3]:
import pandas as pd

# Construct the full path to the CSV file
file_path = os.path.join(path, 'Breast_Cancer.csv')

# Load the dataset
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
print(df.head())

# Display information about the DataFrame
print(df.info())

   Age   Race Marital Status T Stage  N Stage 6th Stage  \
0   68  White        Married       T1      N1       IIA   
1   50  White        Married       T2      N2      IIIA   
2   58  White       Divorced       T3      N3      IIIC   
3   58  White        Married       T1      N1       IIA   
4   47  White        Married       T2      N1       IIB   

               differentiate Grade   A Stage  Tumor Size Estrogen Status  \
0      Poorly differentiated     3  Regional           4        Positive   
1  Moderately differentiated     2  Regional          35        Positive   
2  Moderately differentiated     2  Regional          63        Positive   
3      Poorly differentiated     3  Regional          18        Positive   
4      Poorly differentiated     3  Regional          41        Positive   

  Progesterone Status  Regional Node Examined  Reginol Node Positive  \
0            Positive                      24                      1   
1            Positive                      1

In [4]:
# Map 'Status' to numerical values: Alive (0), Dead (1)
df['Status'] = df['Status'].map({'Alive': 0, 'Dead': 1})

# Identify categorical columns (excluding 'Status' which is now numerical)
categorical_cols = df.select_dtypes(include='object').columns

# Apply one-hot encoding to categorical features
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Display the first 5 rows of the encoded DataFrame
print(df_encoded.head())

# Display information about the encoded DataFrame
print(df_encoded.info())

   Age  Tumor Size  Regional Node Examined  Reginol Node Positive  \
0   68           4                      24                      1   
1   50          35                      14                      5   
2   58          63                      14                      7   
3   58          18                       2                      1   
4   47          41                       3                      1   

   Survival Months  Status  Race_Other  Race_White  Marital Status_Married  \
0               60       0       False        True                    True   
1               62       0       False        True                    True   
2               75       0       False        True                   False   
3               84       0       False        True                    True   
4               50       0       False        True                    True   

   Marital Status_Separated  ...  6th Stage_IIIC  \
0                     False  ...           False   
1           

In [5]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_encoded.drop('Status', axis=1)
y = df_encoded['Status']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the resulting datasets to verify
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (3219, 29)
X_test shape: (805, 29)
y_train shape: (3219,)
y_test shape: (805,)


In [6]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Identify numerical columns for scaling (all columns except boolean and 'Status')
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# Initialize the StandardScaler
scaler = StandardScaler()

# Scale the numerical features in both training and testing sets
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Build the deep learning model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,209 (55.50 KB)

 Trainable params: 14,209 (55.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8474 - loss: 0.4086 - val_accuracy: 0.8618 - val_loss: 0.3403
Epoch 2/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8757 - loss: 0.3208 - val_accuracy: 0.8758 - val_loss: 0.3221
Epoch 3/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8812 - loss: 0.3136 - val_accuracy: 0.8804 - val_loss: 0.3175
Epoch 4/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8878 - loss: 0.3023 - val_accuracy: 0.8913 - val_loss: 0.3143
Epoch 5/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8951 - loss: 0.3005 - val_accuracy: 0.8851 - val_loss: 0.3238
Epoch 6/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8971 - loss: 0.2947 - val_accuracy: 0.8882 - val_loss: 0.3121
Epoch 7/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8963 - loss: 0.2893 - val_accuracy: 0.8851 - val_loss: 0.3131
Epoch 8/50
81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8963 - loss: 0.2861 - val_accuracy: 0.8866 - val_loss:

In [7]:
import joblib

# Save the trained model
model.save('breast_cancer_model.keras')
print("Keras model saved successfully to 'breast_cancer_model.keras'")

# Save the StandardScaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved successfully to 'scaler.pkl'")

Keras model saved successfully to 'breast_cancer_model.keras'
Scaler saved successfully to 'scaler.pkl'


In [8]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import tensorflow as tf

# Load the trained Keras model and scaler
model = tf.keras.models.load_model('breast_cancer_model.keras')
scaler = joblib.load('scaler.pkl')

# --- Streamlit App ----
st.title('Breast Cancer Status Prediction')
st.write('Enter patient details to predict breast cancer status (Alive/Dead).')

# Define input fields based on the preprocessed DataFrame's columns (excluding 'Status')
# This assumes the order and names of columns in X are consistent with the original df_encoded.
# It's crucial that the input features match the features used for training.

# To get the exact list of features from X_train during development:
# print(X_train.columns)

# Example feature list (adjust based on actual X_train.columns output)
# This is a critical part, ensure these match the exact column names after one-hot encoding.

# Numerical inputs
age = st.slider('Age', 20, 90, 50)
tumor_size = st.slider('Tumor Size', 1, 200, 30)
regional_node_examined = st.slider('Regional Node Examined', 0, 50, 10)
reginol_node_positive = st.slider('Regional Node Positive', 0, 20, 2)
survival_months = st.slider('Survival Months', 1, 200, 60)

# Categorical inputs (using selectbox for options)
race = st.selectbox('Race', ['White', 'Black', 'Other'])
marital_status = st.selectbox('Marital Status', ['Married', 'Divorced', 'Single', 'Widowed', 'Separated'])
t_stage = st.selectbox('T Stage', ['T1', 'T2', 'T3', 'T4'])
n_stage = st.selectbox('N Stage', ['N1', 'N2', 'N3'])
sixth_stage = st.selectbox('6th Stage', ['IIA', 'IIIA', 'IIIC', 'IIB', 'I', 'IIIB'])
differentiate = st.selectbox('Differentiate', ['Poorly differentiated', 'Moderately differentiated', 'Well differentiated', 'Undifferentiated'])
grade = st.selectbox('Grade', ['1', '2', '3', ' anaplastic; Grade IV'])
a_stage = st.selectbox('A Stage', ['Regional', 'Distant'])
estrogen_status = st.selectbox('Estrogen Status', ['Positive', 'Negative'])
progesterone_status = st.selectbox('Progesterone Status', ['Positive', 'Negative'])

# Create a dictionary for the input features
input_data = {
    'Age': age,
    'Tumor Size': tumor_size,
    'Regional Node Examined': regional_node_examined,
    'Reginol Node Positive': reginol_node_positive,
    'Survival Months': survival_months,
    'Race_Other': 0,
    'Race_White': 0,
    'Marital Status_Married': 0,
    'Marital Status_Separated': 0,
    'Marital Status_Single': 0,
    'Marital Status_Widowed': 0,
    'T Stage _T2': 0,
    'T Stage _T3': 0,
    'T Stage _T4': 0,
    'N Stage_N2': 0,
    'N Stage_N3': 0,
    '6th Stage_IIB': 0,
    '6th Stage_IIIA': 0,
    '6th Stage_IIIB': 0,
    '6th Stage_IIIC': 0,
    'differentiate_Poorly differentiated': 0,
    'differentiate_Undifferentiated': 0,
    'differentiate_Well differentiated': 0,
    'Grade_1': 0,
    'Grade_2': 0,
    'Grade_3': 0,
    'A Stage_Regional': 0,
    'Estrogen Status_Positive': 0,
    'Progesterone Status_Positive': 0
}

# Handle one-hot encoding for categorical variables
if race == 'Other':
    input_data['Race_Other'] = 1
elif race == 'White':
    input_data['Race_White'] = 1

if marital_status == 'Married':
    input_data['Marital Status_Married'] = 1
elif marital_status == 'Separated':
    input_data['Marital Status_Separated'] = 1
elif marital_status == 'Single':
    input_data['Marital Status_Single'] = 1
elif marital_status == 'Widowed':
    input_data['Marital Status_Widowed'] = 1

if t_stage == 'T2':
    input_data['T Stage _T2'] = 1
elif t_stage == 'T3':
    input_data['T Stage _T3'] = 1
elif t_stage == 'T4':
    input_data['T Stage _T4'] = 1

if n_stage == 'N2':
    input_data['N Stage_N2'] = 1
elif n_stage == 'N3':
    input_data['N Stage_N3'] = 1

if sixth_stage == 'IIB':
    input_data['6th Stage_IIB'] = 1
elif sixth_stage == 'IIIA':
    input_data['6th Stage_IIIA'] = 1
elif sixth_stage == 'IIIB':
    input_data['6th Stage_IIIB'] = 1
elif sixth_stage == 'IIIC':
    input_data['6th Stage_IIIC'] = 1

if differentiate == 'Poorly differentiated':
    input_data['differentiate_Poorly differentiated'] = 1
elif differentiate == 'Undifferentiated':
    input_data['differentiate_Undifferentiated'] = 1
elif differentiate == 'Well differentiated':
    input_data['differentiate_Well differentiated'] = 1

if grade == '1':
    input_data['Grade_1'] = 1
elif grade == '2':
    input_data['Grade_2'] = 1
elif grade == '3':
    input_data['Grade_3'] = 1

if a_stage == 'Regional':
    input_data['A Stage_Regional'] = 1

if estrogen_status == 'Positive':
    input_data['Estrogen Status_Positive'] = 1

if progesterone_status == 'Positive':
    input_data['Progesterone Status_Positive'] = 1


# Create a DataFrame from the input data
input_df = pd.DataFrame([input_data])

# Ensure the order of columns matches X_train
# This is crucial! You must ensure `input_df` columns are in the same order as `X_train`.
# A robust way is to reindex `input_df` using `X_train.columns`.
# For this, X_train.columns must be available, which they are not in the Streamlit app context.
# A better way is to pass X_train.columns to the Streamlit app or hardcode them if they are static.
# For now, let's assume the dictionary creation order matches X_train.columns for simplicity,
# but in a real-world scenario, you'd save X_train.columns and load them.

# The numerical columns that were scaled
numerical_cols = ['Age', 'Tumor Size', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months']

# Scale the numerical features using the loaded scaler
input_df[numerical_cols] = scaler.transform(input_df[numerical_cols])

# Predict button
if st.button('Predict Breast Cancer Status'):
    prediction_proba = model.predict(input_df)[0][0]
    prediction_class = 1 if prediction_proba > 0.5 else 0 # 1 for Dead, 0 for Alive

    st.subheader('Prediction Result:')
    if prediction_class == 0:
        st.success(f'The model predicts the patient is **Alive** (Probability: {prediction_proba:.2f})')
    else:
        st.error(f'The model predicts the patient is **Dead** (Probability: {prediction_proba:.2f})')

    st.write('---')
    st.write('Disclaimer: This is a predictive model and should not be used as medical advice.')

Writing app.py


In [16]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [17]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &




2026-08-24 17:24:30.200 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.70.79.246:8501

2026-08-24 17:24:39.336310: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
  Stopping...


In [15]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import tensorflow as tf

# --- Page Configuration ---
st.set_page_config(
    page_title="Breast Cancer Prediction App",
    page_icon="🎗️",
    layout="centered",
    initial_sidebar_state="expanded",
)

# --- Custom CSS for a more attractive design ---
st.markdown("""
<style>
.main-header {color: #FF6347; text-align: center; font-size: 2.5em; margin-bottom: 20px;}
.sidebar .sidebar-content {background-color: #f0f2f6;}
.stButton>button {background-color: #4CAF50; color: white; border-radius: 5px; padding: 10px 20px; font-size: 1.2em;}
.stButton>button:hover {background-color: #45a049;}
.prediction-success {background-color: #d4edda; color: #155724; border-radius: 5px; padding: 15px; margin-top: 20px;}
.prediction-error {background-color: #f8d7da; color: #721c24; border-radius: 5px; padding: 15px; margin-top: 20px;}
</style>
""", unsafe_allow_html=True)

# Load the trained Keras model and scaler
@st.cache_resource
def load_resources():
    model = tf.keras.models.load_model('breast_cancer_model.keras')
    scaler = joblib.load('scaler.pkl')
    return model, scaler

model, scaler = load_resources()

# --- Main Content ---
st.markdown('<h1 class="main-header">🎗️ Breast Cancer Status Predictor 🎗️</h1>', unsafe_allow_html=True)
st.write('---')
st.write('### ✨ Empowering Health Decisions with AI ✨')
st.markdown("""
This application uses a deep learning model to predict breast cancer status (Alive or Dead)
based on various patient characteristics. Please fill in the details in the sidebar to get a prediction.
""")
st.write('---')

# --- Sidebar for Inputs ---
with st.sidebar:
    st.header('📊 Patient Data Input 📊')
    st.markdown('Please adjust the parameters below:')

    # Numerical inputs
    age = st.slider('Age (years)', 20, 90, 50, help="Patient's age in years.")
    tumor_size = st.slider('Tumor Size (mm)', 1, 200, 30, help="Size of the tumor in millimeters.")
    regional_node_examined = st.slider('Regional Node Examined', 0, 50, 10, help="Number of regional lymph nodes examined.")
    reginol_node_positive = st.slider('Regional Node Positive', 0, 20, 2, help="Number of regional lymph nodes found to be positive.")
    survival_months = st.slider('Survival Months', 1, 200, 60, help="Number of months the patient has survived.")

    st.subheader('Categorical Features 📋')
    # Categorical inputs (using selectbox for options)
    race = st.selectbox('Race', ['White', 'Black', 'Other'])
    marital_status = st.selectbox('Marital Status', ['Married', 'Divorced', 'Single', 'Widowed', 'Separated'])
    t_stage = st.selectbox('T Stage', ['T1', 'T2', 'T3', 'T4'], help="Tumor size and extent (T1-T4).")
    n_stage = st.selectbox('N Stage', ['N1', 'N2', 'N3'], help="Lymph node involvement (N0-N3).")
    sixth_stage = st.selectbox('6th Stage', ['IIA', 'IIIA', 'IIIC', 'IIB', 'I', 'IIIB'], help="AJCC 6th Edition Stage.")
    differentiate = st.selectbox('Differentiation Grade', ['Poorly differentiated', 'Moderately differentiated', 'Well differentiated', 'Undifferentiated'], help="How much the cancer cells resemble normal cells.")
    grade = st.selectbox('Histologic Grade', ['1', '2', '3', ' anaplastic; Grade IV'], help="Grade of the tumor.")
    a_stage = st.selectbox('A Stage', ['Regional', 'Distant'], help="Regional or distant metastasis.")
    estrogen_status = st.selectbox('Estrogen Status', ['Positive', 'Negative'], help="Estrogen receptor status.")
    progesterone_status = st.selectbox('Progesterone Status', ['Positive', 'Negative'], help="Progesterone receptor status.")

# Create a dictionary for the input features with default values matching X_train's columns and order
# It's critical to ensure this matches the order and presence of columns in the training data.
# For a robust solution, X_train.columns should be saved and loaded.
input_data = {
    'Age': age,
    'Tumor Size': tumor_size,
    'Regional Node Examined': regional_node_examined,
    'Reginol Node Positive': reginol_node_positive,
    'Survival Months': survival_months,
    'Race_Other': 0,
    'Race_White': 0,
    'Marital Status_Married': 0,
    'Marital Status_Separated': 0,
    'Marital Status_Single': 0,
    'Marital Status_Widowed': 0,
    'T Stage _T2': 0,
    'T Stage _T3': 0,
    'T Stage _T4': 0,
    'N Stage_N2': 0,
    'N Stage_N3': 0,
    '6th Stage_IIB': 0,
    '6th Stage_IIIA': 0,
    '6th Stage_IIIB': 0,
    '6th Stage_IIIC': 0,
    'differentiate_Poorly differentiated': 0,
    'differentiate_Undifferentiated': 0,
    'differentiate_Well differentiated': 0,
    'Grade_1': 0,
    'Grade_2': 0,
    'Grade_3': 0,
    'A Stage_Regional': 0,
    'Estrogen Status_Positive': 0,
    'Progesterone Status_Positive': 0
}

# Handle one-hot encoding for categorical variables dynamically
if race == 'Other': input_data['Race_Other'] = 1
elif race == 'White': input_data['Race_White'] = 1

if marital_status == 'Married': input_data['Marital Status_Married'] = 1
elif marital_status == 'Separated': input_data['Marital Status_Separated'] = 1
elif marital_status == 'Single': input_data['Marital Status_Single'] = 1
elif marital_status == 'Widowed': input_data['Marital Status_Widowed'] = 1

if t_stage == 'T2': input_data['T Stage _T2'] = 1
elif t_stage == 'T3': input_data['T Stage _T3'] = 1
elif t_stage == 'T4': input_data['T Stage _T4'] = 1

if n_stage == 'N2': input_data['N Stage_N2'] = 1
elif n_stage == 'N3': input_data['N Stage_N3'] = 1

if sixth_stage == 'IIB': input_data['6th Stage_IIB'] = 1
elif sixth_stage == 'IIIA': input_data['6th Stage_IIIA'] = 1
elif sixth_stage == 'IIIB': input_data['6th Stage_IIIB'] = 1
elif sixth_stage == 'IIIC': input_data['6th Stage_IIIC'] = 1

if differentiate == 'Poorly differentiated': input_data['differentiate_Poorly differentiated'] = 1
elif differentiate == 'Undifferentiated': input_data['differentiate_Undifferentiated'] = 1
elif differentiate == 'Well differentiated': input_data['differentiate_Well differentiated'] = 1

if grade == '1': input_data['Grade_1'] = 1
elif grade == '2': input_data['Grade_2'] = 1
elif grade == '3': input_data['Grade_3'] = 1

if a_stage == 'Regional': input_data['A Stage_Regional'] = 1

if estrogen_status == 'Positive': input_data['Estrogen Status_Positive'] = 1

if progesterone_status == 'Positive': input_data['Progesterone Status_Positive'] = 1


# Create a DataFrame from the input data
# Ensure the columns are in the correct order as per training data
# A more robust solution would involve saving X_train.columns and reindexing here.
input_df = pd.DataFrame([input_data])

# Numerical columns that were scaled during training
numerical_cols = ['Age', 'Tumor Size', 'Regional Node Examined', 'Reginol Node Positive', 'Survival Months']

# Scale the numerical features using the loaded scaler
input_df[numerical_cols] = scaler.transform(input_df[numerical_cols])

# Predict button
st.write('---')
if st.button('🚀 Predict Breast Cancer Status'):
    prediction_proba = model.predict(input_df)[0][0]
    prediction_class = 1 if prediction_proba > 0.5 else 0 # 1 for Dead, 0 for Alive

    st.subheader('🔮 Prediction Result:')
    if prediction_class == 0:
        st.markdown(f'<div class="prediction-success">✅ The model predicts the patient is **Alive** (Probability: {prediction_proba:.2f})</div>', unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="prediction-error">⚠️ The model predicts the patient is **Dead** (Probability: {prediction_proba:.2f})</div>', unsafe_allow_html=True)

    st.write('---')
    st.info('💡 Disclaimer: This is a predictive model based on historical data and should not be used as medical advice. Always consult with a healthcare professional for diagnosis and treatment.')


Overwriting app.py
